In [1]:
import cv2
from skimage.feature import local_binary_pattern, hog, graycomatrix, graycoprops
import numpy as np
import pandas as pd
import os
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import openpyxl  # For creating Excel files

def read_image(file_path):
    """Reads an image from the given file path."""
    return cv2.imread(file_path)

def convert_to_gray(image):
    """Converts an image to grayscale."""
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

def gaussian_filtering(gray_image):
    """Applies Gaussian filtering to reduce noise."""
    return cv2.GaussianBlur(gray_image, (5, 5), 0)

def histogram_stretching(gray_image):
    """Stretches the histogram of a grayscale image to improve contrast."""
    min_value = np.min(gray_image)
    max_value = np.max(gray_image)
    stretched = (gray_image - min_value) / (max_value - min_value) * 255
    return stretched.astype(np.uint8)

def extract_lbp_histogram(image, P=8, R=1, method='uniform'):
    """Extracts the LBP histogram from the image."""
    lbp = local_binary_pattern(image, P, R, method=method)
    n_bins = int(lbp.max() + 1)
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, density=True)
    return hist

def extract_hog_features(image, orientations=8, pixels_per_cell=(16, 16), cells_per_block=(1, 1)):
    """Extracts HOG features from the image."""
    features = hog(image, orientations=orientations, pixels_per_cell=pixels_per_cell,
                   cells_per_block=cells_per_block, feature_vector=True)
    return features

def extract_glcm_features(image, distances=[5], angles=[0], levels=256, symmetric=True, normed=True):
    """Extracts GLCM features from the image."""
    glcm = graycomatrix(image, distances=distances, angles=angles, levels=levels,
                        symmetric=symmetric, normed=normed)
    if glcm.size == 0:
        return np.zeros(5)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    return np.array([contrast, dissimilarity, homogeneity, energy, correlation])

def extract_statistical_features(image):
    """Extracts statistical features from the image."""
    mean_val = np.mean(image)
    std_val = np.std(image)
    var_val = np.var(image)
    skew_val = skew(image.ravel())
    kurt_val = kurtosis(image.ravel())
    entropy_val = entropy(np.histogram(image.ravel(), bins=256)[0] + 1e-9) # Add small constant to avoid log(0)
    min_val = np.min(image)
    max_val = np.max(image)
    return np.array([mean_val, std_val, var_val, skew_val, kurt_val, entropy_val, min_val, max_val])

def extract_features(image, feature_type='combined'):
    """Extracts features from the image based on the specified type."""
    if feature_type == 'lbp':
        return extract_lbp_histogram(image)
    elif feature_type == 'hog':
        return extract_hog_features(image)
    elif feature_type == 'glcm':
        return extract_glcm_features(image)
    elif feature_type == 'statistical':
        return extract_statistical_features(image)
    elif feature_type == 'combined':
        lbp_hist = extract_lbp_histogram(image)
        hog_feats = extract_hog_features(image)
        glcm_feats = extract_glcm_features(image)
        stat_feats = extract_statistical_features(image)
        return np.concatenate((lbp_hist, hog_feats, glcm_feats, stat_feats))
    else:
        raise ValueError(f"Invalid feature type: {feature_type}")

def load_and_process_images(folder_path, feature_type):
    """Loads and processes all images from the given folder and extracts features."""
    image_files = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
    images = []
    labels = []
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        original_image = read_image(image_path)
        if original_image is None:
            print(f"Could not read image: {image_file}")
            continue
        gray_image = convert_to_gray(original_image)
        filtered_image = gaussian_filtering(gray_image)
        stretched_image = histogram_stretching(filtered_image)
        images.append(stretched_image)

        # Extract label from folder name.  Assumes folder structure like 'CXR'
        label = os.path.basename(folder_path)
        labels.append(label)

    features = [extract_features(image, feature_type=feature_type) for image in images]
    return images, image_files, np.array(features), labels

def create_excel_file(df, filename="features.xlsx"):
    """Creates an Excel file from a Pandas DataFrame."""
    df.to_excel(filename, index=False, engine='openpyxl')
    print(f"Excel file '{filename}' created successfully.")

if __name__ == '__main__':
    # 1. Load and Process Images
    folder_path = '/kaggle/input/medical-mnist/CXR' # Change this path
    feature_type = 'combined'  # Can be 'lbp', 'hog', 'glcm', 'statistical', or 'combined'
    images, image_files, features, labels = load_and_process_images(folder_path, feature_type=feature_type)

    # 2. Create DataFrame
    feature_names = []
    if feature_type == 'lbp':
        feature_names = [f'lbp_{i}' for i in range(features.shape[1])]
    elif feature_type == 'hog':
        feature_names = [f'hog_{i}' for i in range(features.shape[1])]
    elif feature_type == 'glcm':
        feature_names = [f'glcm_{i}' for i in range(features.shape[1])]
    elif feature_type == 'statistical':
        feature_names = ['mean', 'std', 'variance', 'skewness', 'kurtosis', 'entropy', 'min_val', 'max_val']
    elif feature_type == 'combined':
        lbp_len = len(extract_lbp_histogram(images[0]))
        hog_len = len(extract_hog_features(images[0]))
        glcm_len = 5
        stat_len = 8
        feature_names = [f'lbp_{i}' for i in range(lbp_len)] + \
                        [f'hog_{i}' for i in range(hog_len)] + \
                        [f'glcm_{i}' for i in range(glcm_len)] + \
                        ['mean', 'std', 'variance', 'skewness', 'kurtosis', 'entropy', 'min_val', 'max_val']

    df_features = pd.DataFrame(features, columns=feature_names)
    df_features['label'] = labels  # Add the labels

    # 3. Print the first few rows of the DataFrame
    print("Extracted Feature Table (first 5 rows):")
    print(df_features.head())
    print("\nShape of the Feature Table:", df_features.shape)

    # 4. Create Excel File
    create_excel_file(df_features, filename="cxr_features.xlsx")


Extracted Feature Table (first 5 rows):
      lbp_0     lbp_1     lbp_2     lbp_3     lbp_4     lbp_5     lbp_6  \
0  0.002713  0.023329  0.024957  0.132650  0.622559  0.227051  0.031467   
1  0.001899  0.024414  0.033095  0.131293  0.550944  0.235460  0.046658   
2  0.005154  0.033908  0.033095  0.107422  0.560710  0.236545  0.041504   
3  0.003526  0.033366  0.024685  0.134277  0.595161  0.214572  0.038249   
4  0.003798  0.037164  0.041775  0.114475  0.550944  0.225152  0.051812   

      lbp_7     lbp_8     lbp_9  ...    glcm_4        mean        std  \
0  0.010851  0.018175  0.017361  ...  0.671644  138.884521  70.585578   
1  0.022786  0.039605  0.024957  ...  0.681078  171.974121  65.411687   
2  0.026313  0.042860  0.023600  ...  0.701361  173.150146  65.433181   
3  0.018175  0.026584  0.022515  ...  0.640100  153.961670  71.903508   
4  0.023058  0.034722  0.028212  ...  0.657897  166.888672  70.768972   

      variance  skewness  kurtosis   entropy  min_val  max_val  label 